## Prediction for Step 7 (from reading the script, before running)
- Signature matches run_pipeline.py step7 (including keep_clips).
- clip_name: ".wav" is stripped from all four tables before merging, so Step 4's mismatch is handled here.
- Weather columns: Step 7 selects "Temperature", "Windspeed", "Humidity"; our Step 6 output has lowercase names. Expect KeyError at the weather merge when run as-is.
- After a notebook workaround (renamed copy of the weather table):
  - 1,747 rows. The 599 clips with blank weather get NaN Temperature/Windspeed/Humidity and NaN Volume_Wind_Ratio (no fill in the code).
  - Only Buff-rumped Warbler, Black-cheeked Ant-Tanager and Blue-black Grosbeak flags can be 1; the other 7 are all 0. At most 66 clips have a sentinel, so Eerie_Silence = 1 for at least 1,681 of 1,747.
  - Volume_Spike_15s: rows sorted by Recorder and time; the first clip is 0.
  - keep_clips: with all 1,747 meaningful, all rows are kept.
- The saved CSV has more columns than the 29 model features (clip_name, species, Recorder, Timestamp, Datetime also appear).

In [1]:
import sys, ast
from pathlib import Path
import pandas as pd

HOME = Path.home()
OUTPUTS_DIR  = HOME / "kwf_test" / "outputs"
PIPELINE_DIR = Path("/Users/huyguy/Documents/Documents - Huys MacBook Pro/Job Materials/KwF/Inference Pipeline/KWF_Human_Presence/inference_pipeline")
MODELS_DIR   = PIPELINE_DIR / "models"
sys.path.insert(0, str(PIPELINE_DIR))
sys.path.insert(0, str(PIPELINE_DIR / "scripts"))

import feature_engineering as fe

labels = [l.strip() for l in open(MODELS_DIR / "birdnet_fp16_pruned493.labels.txt")]
allow  = {l.strip() for l in open(MODELS_DIR / "geomodel_allowlist_218.txt") if l.strip()}
bird   = pd.read_csv(OUTPUTS_DIR / "nb03_birdnet_output.csv")
detected = pd.Series([s for lst in bird["species"].apply(ast.literal_eval) for s in lst]).value_counts()

rows = [{"sentinel": s, "in labels": s in labels, "in allow-list": s in allow,
         "detections (recorder 5)": int(detected.get(s, 0))} for s in fe.SENTINEL_SPECIES]
print(pd.DataFrame(rows).to_string(index=False))

                                      sentinel  in labels  in allow-list  detections (recorder 5)
    Myiothlypis fulvicauda_Buff-rumped Warbler       True           True                       58
Habia atrimaxillaris_Black-cheeked Ant-Tanager       True           True                        6
  Thamnophilus bridgesi_Black-hooded Antshrike       True           True                        0
                   Tinamus major_Great Tinamou       True           True                        0
  Patagioenas nigrirostris_Short-billed Pigeon       True           True                        0
    Ramphastos ambiguus_Yellow-throated Toucan       True           True                        0
      Cyanoloxia cyanoides_Blue-black Grosbeak       True           True                        2
                 Lipaugus unirufus_Rufous Piha       True           True                        0
      Threnetes ruckeri_Band-tailed Barbthroat       True           True                        0
                    

What this does: sets up paths, imports feature_engineering, and checks each of the 10 sentinel species against the BirdNET labels file, the allow-list, and our recorder 5 detections.

How to read the output: “in labels” False means that spelling can never match a BirdNET output, so the flag is always 0. “in allow-list” False means BirdNET drops that species before Step 7 ever sees it, so the flag is always 0 too. The detections column should show 58, 6 and 2 for the three species and 0 for the rest. A sentinel that’s missing from either list would be a finding for the team, because the model would be given a feature that can never fire.

**Sentinel check (passes):** all 10 sentinel species are present in the BirdNET labels file and the allow-list, so every flag can fire. On recorder 5, only 3 are detected (58, 6 and 2 detections); the other 7 are all zeros.

In [2]:
cnn = pd.read_csv(OUTPUTS_DIR / "nb02_recorder5_rerun_predictions.csv")
meaningful = cnn[cnn["predicted_label"] == "meaningful"]["clip_name"].tolist()
print("Meaningful clips passed as keep_clips:", len(meaningful))

try:
    feats_asis = fe.run_feature_engineering(
        acoustic_csv=str(OUTPUTS_DIR / "nb04_recorder5_acoustic_features.csv"),
        birdnet_csv=str(OUTPUTS_DIR / "nb03_birdnet_output.csv"),
        metadata_csv=str(OUTPUTS_DIR / "nb05_metadata.csv"),
        weather_csv=str(OUTPUTS_DIR / "nb06_weather_matches.csv"),
        output_csv=str(OUTPUTS_DIR / "nb07_features_asis.csv"),
        keep_clips=meaningful,
    )
except Exception as e:
    print("\nStep 7 FAILED:", type(e).__name__, e)

Meaningful clips passed as keep_clips: 1747
Loading inputs...
Merging inputs...

Step 7 FAILED: KeyError "['Temperature', 'Windspeed', 'Humidity'] not in index"


What this does: calls run_feature_engineering with our Step 3, 4, 5 and 6 outputs and the keep_clips list from Step 2, wrapped in try/except so a crash is recorded and doesn’t stop the notebook.

How to read the output: Predicted a KeyError naming Temperature, Windspeed and Humidity, because Step 7 selects those capitalised names and our Step 6 table has temperature, windspeed and humidity. If that appears, Step 7 can’t finish as written. If it prints “Done”, my reading was wrong, and we inspect the output.

**Finding 1 (confirmed):** `run_feature_engineering` fails with `KeyError: ['Temperature', 'Windspeed', 'Humidity'] not in index`. Step 7 selects capitalised weather columns, but Step 6's output has lowercase `temperature`, `windspeed`, `humidity`. Step 7 cannot finish on Step 6's real output.

In [3]:
wm = pd.read_csv(OUTPUTS_DIR / "nb06_weather_matches.csv")
wm = wm.rename(columns={"temperature": "Temperature", "windspeed": "Windspeed", "humidity": "Humidity"})
wm.to_csv(OUTPUTS_DIR / "nb07_weather_renamed.csv", index=False)

try:
    feats = fe.run_feature_engineering(
        acoustic_csv=str(OUTPUTS_DIR / "nb04_recorder5_acoustic_features.csv"),
        birdnet_csv=str(OUTPUTS_DIR / "nb03_birdnet_output.csv"),
        metadata_csv=str(OUTPUTS_DIR / "nb05_metadata.csv"),
        weather_csv=str(OUTPUTS_DIR / "nb07_weather_renamed.csv"),
        output_csv=str(OUTPUTS_DIR / "nb07_features.csv"),
        keep_clips=meaningful,
    )
except Exception as e:
    print("\nStep 7 FAILED:", type(e).__name__, e)

Loading inputs...
Merging inputs...
Encoding sentinel species...
Parsing confidence...
Engineering Patrick's features...
Kept 1,747 AED-meaningful clips.
Saving output...
Done. 1,747 rows saved to /Users/huyguy/kwf_test/outputs/nb07_features.csv


What this does: writes a copy of Step 6’s matched table with Temperature, Windspeed and Humidity capitalised, and runs run_feature_engineering on it with the same keep_clips list.

How to read the output: “Done. 1,747 rows saved” means the workaround got past the mismatch and all clips were kept (all 1,747 are meaningful). Any new error is the next finding.

In [4]:
model_features = (
    ["RMS_Energy", "Spectral_Contrast", "Spectral_Flatness", "Spectral_Bandwidth", "Spectral_Rolloff_85",
     "Onset_Strength", "MFCC_8", "MFCC_9", "MFCC_12", "MFCC_13", "confidence"]
    + fe.SENTINEL_SPECIES
    + ["Eerie_Silence", "hour_sin", "hour_cos", "Volume_Wind_Ratio", "Volume_Spike_15s",
       "Temperature", "Windspeed", "Humidity"]
)
print("Shape:", feats.shape, "| model features expected:", len(model_features))
print("Missing model-feature columns:", [c for c in model_features if c not in feats.columns] or "none")
print("Extra columns:", [c for c in feats.columns if c not in model_features])

nulls = feats[model_features].isna().sum()
print("\nBlanks per feature (non-zero only):\n", nulls[nulls > 0].to_string() if (nulls > 0).any() else "none")

print("\nSentinel flag sums (non-zero only):")
s = feats[fe.SENTINEL_SPECIES].sum()
print(s[s > 0].to_string())
print("Eerie_Silence = 1:", int(feats["Eerie_Silence"].sum()), "of", len(feats))

print("\nVolume_Spike_15s:", feats["Volume_Spike_15s"].describe()[["min", "mean", "max"]].round(4).to_dict())
print("First clip:", feats["clip_name"].iloc[0], "| spike:", feats["Volume_Spike_15s"].iloc[0])
print("hour_sin range:", round(feats["hour_sin"].min(), 3), "to", round(feats["hour_sin"].max(), 3))
print("Sorted by time:", feats["Datetime"].is_monotonic_increasing)
print("Example clip_name:", feats["clip_name"].iloc[0])

Shape: (1747, 34) | model features expected: 29
Missing model-feature columns: none
Extra columns: ['clip_name', 'species', 'Recorder', 'Timestamp', 'Datetime']

Blanks per feature (non-zero only):
 Volume_Wind_Ratio    599
Temperature          599
Windspeed            599
Humidity             599

Sentinel flag sums (non-zero only):
Myiothlypis fulvicauda_Buff-rumped Warbler        58
Habia atrimaxillaris_Black-cheeked Ant-Tanager     6
Cyanoloxia cyanoides_Blue-black Grosbeak           2
Eerie_Silence = 1: 1681 of 1747

Volume_Spike_15s: {'min': 0.0, 'mean': 0.0016, 'max': 0.2493}
First clip: Audio_Moth_5_20250319_163237 | spike: 0.0
hour_sin range: -0.966 to -0.866
Sorted by time: True
Example clip_name: Audio_Moth_5_20250319_163237


What this does: checks the output against the prediction: row count, whether all 29 model features exist, blanks per column, the sentinel flags and Eerie_Silence, Volume_Spike_15s, and hour_sin / hour_cos.

How to read the output: 1,747 rows means no clip was lost in the merges. The blank show 599 for the three weather columns and for Volume_Wind_Ratio, and none elsewhere. Only three sentinel columns have nonzero sums (58, 6, 2). Eerie_Silence is 1 for at least 1,681 clips. Volume_Spike_15s is 0 for the first clip and never negative. A blank in a model feature matters, because scikit-learn models usually refuse blanks.

In [5]:
drop = set(meaningful[::100])
keep = [c for c in meaningful if c not in drop]

feats_f = fe.run_feature_engineering(
    acoustic_csv=str(OUTPUTS_DIR / "nb04_recorder5_acoustic_features.csv"),
    birdnet_csv=str(OUTPUTS_DIR / "nb03_birdnet_output.csv"),
    metadata_csv=str(OUTPUTS_DIR / "nb05_metadata.csv"),
    weather_csv=str(OUTPUTS_DIR / "nb07_weather_renamed.csv"),
    output_csv=str(OUTPUTS_DIR / "nb07_features_filtered.csv"),
    keep_clips=keep,
)
dropped_stripped = {c.replace(".wav", "") for c in drop}
print("\nRows:", len(feats_f), "| dropped:", len(drop))
print("Dropped clips still present:", len(dropped_stripped & set(feats_f["clip_name"])))
a = feats.set_index("clip_name")["Volume_Spike_15s"]
b = feats_f.set_index("clip_name")["Volume_Spike_15s"]
print("Max Volume_Spike_15s difference vs unfiltered run:", (a.reindex(b.index) - b).abs().max())

Loading inputs...
Merging inputs...
Encoding sentinel species...
Parsing confidence...
Engineering Patrick's features...
Kept 1,729 AED-meaningful clips.
Saving output...
Done. 1,729 rows saved to /Users/huyguy/kwf_test/outputs/nb07_features_filtered.csv

Rows: 1729 | dropped: 18
Dropped clips still present: 0
Max Volume_Spike_15s difference vs unfiltered run: 0.0


What this does: removes every 100th clip from the keep_clips list, runs Step 7 again, and compares with the full run.

How to read the output: 1,729 rows means 18 clips were dropped. “Dropped clips still present: 0” means the filter works. A max spike difference of 0.0 means Volume_Spike_15s was computed before the filter, so removing clips didn’t change anything.

**Filter test:** removing every 100th clip from `keep_clips` (18 of 1,747) gave 1,729 rows with none of the dropped clips present, and `Volume_Spike_15s` was identical to the unfiltered run (max difference 0.0), confirming the spike feature is computed before filtering. The filter was simulated; recorder 5 has no clips flagged `not_meaningful`.

**Not tested:** the HP model's handling of blanks (Step 8 is a placeholder); multiple recorders; filtering with the real flagged clips (recorder 1); whether these 29 columns match the team's final feature list (`HP_Model/docs/feature_pipeline_progress.md`).

## Observations (Step 7)

**Tested:** `run_feature_engineering` on recorder 5 (1,747 clips) using the Step 3, 4, 5 and 6 outputs and `keep_clips` from Step 2.

**Result:** works after a test workaround. 1,747 rows, 34 columns = 29 model features + 5 extras (clip_name, species, Recorder, Timestamp, Datetime). Sentinel flags as predicted (58, 6, 2; other 7 all zero); `Eerie_Silence` = 1 for 1,681 clips; `Volume_Spike_15s` is 0 for the first clip and never negative; rows are sorted by time. `.wav` is stripped from all tables, so the Step 4 name mismatch is handled here.

**Issues**
1. As run, Step 7 fails with `KeyError: ['Temperature', 'Windspeed', 'Humidity'] not in index`. It selects capitalised weather columns, but Step 6 writes lowercase ones. (Workaround: renamed copy of the weather table; repo files unchanged.)
2. 599 clips (34%) have blank Temperature, Windspeed, Humidity and Volume_Wind_Ratio, because Step 6 found no weather for them. No fill exists. Not tested: whether the HP model accepts blanks.
3. `Eerie_Silence` = 1 for 96% of clips on this recording (only 3 of 10 sentinels detected).
4. `hour_sin`/`hour_cos` use the clip's own hour, while Step 6 matches weather to the nearest hour, so the two can disagree for clips in the second half of an hour.

**Not tested:** the HP model's handling of blanks (Step 8 is a placeholder); multiple recorders; whether these 29 columns match the team's final feature list (`HP_Model/docs/feature_pipeline_progress.md`).

**Questions for the team**
- Should blank weather be filled, or should those clips be dropped?
- Should Step 7 expect lowercase weather columns (as Step 6 writes them), or Step 6 write capitalised ones?
- Is nearest-hour matching in Step 6 intended?